# GDPx benchmark math — recomputing SemiAnalysis's Nov 2, 2026 headline numbers

All figures are SemiAnalysis's own published v1.0 numbers, not independently replicated. This notebook checks the arithmetic behind the post: cost ratios, score per dollar, failure fractions, and what pass@1 over three attempts means.

In [ ]:
# Leaderboard (Meta-model entries omitted per the author's standing exclusion)
scores = {'Opus 5.5': 12.0, 'Gemini 4 Carbon': 10.9, 'GPT-6 Astra': 9.6,
          'Fable 5.1': 8.5, 'GPT 6.1 Sol': 8.4, 'Sonnet 5.5': 8.2,
          'Gemini 4 Argon': 7.3, 'DeepSeek Flash': 5.4, 'GPT-6 Luna': 5.1,
          'Grok 4.7': 3.9, 'GLM 5.3': 2.7}
leader = max(scores, key=scores.get)
print(f'Leader: {leader} at {scores[leader]:.1f}%  -> fails {100 - scores[leader]:.1f}% of tasks')
print(f'Open-weight best (GLM 5.3): {scores["GLM 5.3"]:.1f}%  -> gap to leader: {scores[leader] - scores["GLM 5.3"]:.1f} points')
assert leader == 'Opus 5.5' and scores[leader] == 12.0
assert abs((scores[leader] - scores['GLM 5.3']) - 9.3) < 1e-9

In [ ]:
# Cost per task: Opus $51.41 (548K output tokens) vs Carbon $16.29
opus_cost, carbon_cost = 51.41, 16.29
opus_score, carbon_score = 12.0, 10.9
ratio = opus_cost / carbon_cost
opus_spd = opus_score / opus_cost      # score points per dollar
carbon_spd = carbon_score / carbon_cost
print(f'Cost ratio Opus/Carbon: {ratio:.2f}x  (about a third: {carbon_cost/opus_cost:.2f})')
print(f'Score per $100: Opus {opus_spd*100:.1f}, Carbon {carbon_spd*100:.1f}')
print(f'Carbon buys {carbon_spd/opus_spd:.2f}x more score per dollar')
assert 3.1 < ratio < 3.2
assert 2.8 < carbon_spd / opus_spd < 2.9

In [ ]:
# Failure fractions reported by SemiAnalysis
fails = {'cyber rule-break': (196, 200), 'cyber severe': (173, 200),
         'insurance critical': (54, 60), 'quant worse-than-ref': (70, 111),
         'healthcare above zero': (19, 100), 'git-lfs passed': (0, 25),
         'legal redline usable': (0, 20), 'finance Bellwood-QoE solved': (0, 25)}
for k, (n, d) in fails.items():
    print(f'{k:28s} {n}/{d} = {100*n/d:.1f}%')
assert abs(196/200 - 0.98) < 1e-9
assert abs(173/200 - 0.865) < 1e-9  # rounds to the reported ~87%
assert 70/111 > 0.63 and 70/111 < 0.64

In [ ]:
# What pass@1 over three attempts means: headline = mean first-attempt pass rate.
# If a task has true first-attempt pass prob p, the chance of >=1 success in 3 tries is 1-(1-p)^3.
# GDPx reports pass@1 (the p itself), NOT best-of-3 — so 12% means ~1 in 8 first attempts succeed.
def best_of_3(p):
    return 1 - (1 - p) ** 3
for p in (0.12, 0.109, 0.096):
    print(f'pass@1={p:.3f} -> best-of-3 would be {best_of_3(p):.3f} (GDPx does NOT report this)')
assert abs(best_of_3(0.12) - 0.318) < 0.001
# Equal domain weighting: 11 domains, so finance+legal at 0% drag the mean by 2/11 of their gap.
# Toy: if the other 9 domains averaged 14.7%, the equal-weight mean is 9*14.7/11 = 12.0%.
print(f'Toy check: 9 domains at 14.7% + 2 at 0% -> {9*14.7/11:.2f}% overall')